In [4]:
import requests

api_key = "	3361d688-e3ad-409f-9710-f35cef6fb405"
# Company 00000006 is a standard test case (HSBC)
url = "https://api.company-information.service.gov.uk/company/00000006"

# Pass the key as the username, and an empty string as the password
response = requests.get(url, auth=('3361d688-e3ad-409f-9710-f35cef6fb405', ''))

if response.status_code == 200:
    print("Success!")
    print(response.json()['company_name'])
else:
    print(f"Failed with status code: {response.status_code}")
    print(response.text)

Failed with status code: 401
{"error":"Invalid Authorization","type":"ch:service"}


In [10]:
import requests
import base64

# 1. Configuration
API_KEY = "3a812851-46a4-4641-bea7-5e5d6f853abf"  # Put your key here
COMPANY_NUMBER = "00000006"     # HSBC (Good for testing)
URL = f"https://api.company-information.service.gov.uk/company/00000006"

# 2. Manual Encoding (Ensures the colon is included correctly)
# The format must be "Key:" (Key followed by a colon, then nothing)
auth_string = f"{API_KEY}:"
encoded_auth = base64.b64encode(auth_string.encode('ascii')).decode('ascii')

headers = {
    'Authorization': f'Basic {encoded_auth}'
}

# 3. Execution
try:
    response = requests.get(URL, headers=headers)
    
    if response.status_code == 200:
        print("✅ Success! Key is working.")
        print(f"Company Name: {response.json().get('company_name')}")
    else:
        print(f"❌ Failed with status code: {response.status_code}")
        print(f"Response: {response.text}")

except Exception as e:
    print(f"An error occurred: {e}")

✅ Success! Key is working.
Company Name: MARINE AND GENERAL MUTUAL LIFE ASSURANCE SOCIETY


In [12]:
import requests
import json

# --- CONFIGURATION ---
API_KEY = "3a812851-46a4-4641-bea7-5e5d6f853abf"
COMPANY_NUMBER = "R0000419"  # Example: HSBC
BASE_URL = "https://api.company-information.service.gov.uk"

def get_psc_data(company_num):
    # 1. Endpoint for actual Persons with Significant Control
    psc_url = f"{BASE_URL}/company/{company_num}/persons-with-significant-control"
    
    # 2. Endpoint for PSC Statements (e.g., 'No PSC identified')
    statements_url = f"{BASE_URL}/company/{company_num}/persons-with-significant-control-statements"
    
    auth = (API_KEY, '')
    
    print(f"--- Fetching PSC Data for {company_num} ---")
    
    # Get the Persons
    psc_res = requests.get(psc_url, auth=auth)
    if psc_res.status_code == 200:
        pscs = psc_res.json().get('items', [])
        for p in pscs:
            name = p.get('name', 'N/A')
            kind = p.get('kind', 'N/A')
            # 'natures_of_control' is the most important field
            controls = ", ".join(p.get('natures_of_control', []))
            print(f"[PSC] {name} ({kind})")
            print(f"      Control: {controls}\n")
    elif psc_res.status_code == 404:
        print("No specific PSCs found.")
    
    # Get the Statements
    stat_res = requests.get(statements_url, auth=auth)
    if stat_res.status_code == 200:
        statements = stat_res.json().get('items', [])
        for s in statements:
            print(f"[STATEMENT] {s.get('statement')}")
            print(f"            Notified on: {s.get('notified_on')}\n")

# Run it
get_psc_data(COMPANY_NUMBER)

--- Fetching PSC Data for R0000419 ---
[PSC] Mr Ian William Larmor Webb (individual-person-with-significant-control)
      Control: ownership-of-shares-25-to-50-percent

[PSC] Mr Robert Mitchel Webb (individual-person-with-significant-control)
      Control: significant-influence-or-control



In [13]:
import requests
import json

# --- CONFIG ---
API_KEY = "3a812851-46a4-4641-bea7-5e5d6f853abf"
COMPANY_NUMBER = "R0000419" # John Hogg & Co. Ltd
BASE_URL = "https://api.company-information.service.gov.uk"
AUTH = (API_KEY, '')

def get_full_intel(co_num):
    endpoints = {
        "Profile": f"{BASE_URL}/company/{co_num}",
        "Officers": f"{BASE_URL}/company/{co_num}/officers",
        "PSCs": f"{BASE_URL}/company/{co_num}/persons-with-significant-control",
        "Statements": f"{BASE_URL}/company/{co_num}/persons-with-significant-control-statements",
        "Filings": f"{BASE_URL}/company/{co_num}/filing-history?items_per_page=5"
    }
    
    results = {}
    
    for name, url in endpoints.items():
        res = requests.get(url, auth=AUTH)
        if res.status_code == 200:
            results[name] = res.json()
        else:
            results[name] = f"Error {res.status_code}"

    # --- FORMATTED OUTPUT ---
    print(f"=== DEEP DIVE: {co_num} ===")
    
    # 1. PROFILE (Type & Subtype)
    p = results.get("Profile", {})
    print(f"\n[STRUCTURE]")
    print(f"Name: {p.get('company_name')}")
    print(f"Type: {p.get('type')} / Subtype: {p.get('subtype', 'none')}")
    print(f"Status: {p.get('company_status')}")

    # 2. OFFICERS (The Board)
    print(f"\n[OFFICERS / DIRECTORS]")
    officers = results.get("Officers", {}).get('items', [])
    for o in officers:
        status = "Active" if not o.get('resigned_on') else f"Resigned ({o.get('resigned_on')})"
        print(f"- {o.get('name')} | Role: {o.get('officer_role')} | Status: {status}")

    # 3. OWNERSHIP (PSCs)
    print(f"\n[OWNERSHIP / PSCs]")
    pscs = results.get("PSCs", {}).get('items', [])
    if not pscs: print("No direct PSCs listed.")
    for psc in pscs:
        print(f"- {psc.get('name')} ({psc.get('kind')})")
        print(f"  Natures of Control: {', '.join(psc.get('natures_of_control', []))}")

    # 4. FILING HISTORY (The "Paper Trail")
    print(f"\n[LATEST FILINGS - Where shares are detailed]")
    filings = results.get("Filings", {}).get('items', [])
    for f in filings:
        print(f"- {f.get('date')}: {f.get('description')} ({f.get('type')})")

get_full_intel(COMPANY_NUMBER)

=== DEEP DIVE: R0000419 ===

[STRUCTURE]
Name: JOHN HOGG & CO, LIMITED
Type: ltd / Subtype: none
Status: active

[OFFICERS / DIRECTORS]
- JOHNSTON, Mark | Role: secretary | Status: Active
- JOHNSTON, Mark | Role: director | Status: Active
- WEBB, Ian Kenneth | Role: director | Status: Active
- WEBB, Ian William Larmor | Role: director | Status: Active
- WEBB, Robert Mitchel | Role: director | Status: Active
- WEBB, William Robert | Role: director | Status: Active
- WRIGHT, Steven Paul | Role: director | Status: Active
- CAIRNS, Colin William | Role: secretary | Status: Resigned (2018-08-16)
- CAIRNS, Colin William | Role: director | Status: Resigned (2023-09-30)
- HENDERSON, Arthur Robinson | Role: director | Status: Resigned (2015-04-30)
- WEBB, Andrew Brian | Role: director | Status: Resigned (2023-09-30)
- WEBB, William H | Role: director | Status: Resigned (2009-04-03)

[OWNERSHIP / PSCs]
- Mr Ian William Larmor Webb (individual-person-with-significant-control)
  Natures of Control